# Modeling Experiments.

This notebook summarizes the results generated by `src/models/run_experiments.py`. We will start from the saved artifacts: metrics, summary, predictions, and models to:

1. Visualize and compare metrics by model/fold.
2. Analyze the errors on the test set (residuals, and y_true vs y_pred scatter).
3. Review feature importance (if available for the model) and document conclusions.

### 1. Setup
Update the `EXPERIMENT_DIR` path with the specific folder (e.g., `experiment_YYYYMMDD_HHMMSS`).

In [9]:
import json
from pathlib import Path

import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

EXPERIMENT_BASE_DIR = Path("../data/results/modeling/experiments")
runner_experiments = sorted(
    [d for d in EXPERIMENT_BASE_DIR.iterdir() if d.is_dir() and d.name.startswith("runner_id_")],
    reverse=True,
)
if not runner_experiments:
    raise FileNotFoundError("No runner_id experiments found. Run run_experiments.py first.")
EXPERIMENT_DIR = runner_experiments[0]
print(f"Using experiment directory: {EXPERIMENT_DIR.name}")
metrics_path = EXPERIMENT_DIR / "metrics.csv"
summary_path = EXPERIMENT_DIR / "summary.csv"
predictions_path = EXPERIMENT_DIR / "predictions.parquet"
config_path = EXPERIMENT_DIR / "config.json"
feature_cols_path = EXPERIMENT_DIR / "feature_columns.json"
models_dir = EXPERIMENT_DIR  

metrics_df = pd.read_csv(metrics_path)
summary_df = pd.read_csv(summary_path)
pred_df = pd.read_parquet(predictions_path)
if feature_cols_path.exists():
    feature_columns = json.loads(feature_cols_path.read_text())
else:
    feature_columns = None

if config_path.exists():
    config = json.loads(config_path.read_text())
    target_name = config.get("target", "reported_rpe").lower()
else:
    config = None
    target_name = "reported_rpe"

target_pretty_map = {
    "reported_rpe": "RPE",
    "fatigue_score": "Fatigue Score",
    "fatigue_level": "Fatigue Level",
}
target_label = target_pretty_map.get(target_name, target_name)

summary_df

Using experiment directory: runner_id_20251118_174641


,model,split,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,med_ae_mean,med_ae_std,max_err_mean,max_err_std,samples_total
0,catboost,cv,0.063420,0.011731,0.085994,0.017677,0.645656,0.137556,0.048047,0.010564,0.368016,0.041385,12595
1,catboost,test,0.045876,NaN,0.062259,NaN,0.841095,NaN,0.034661,NaN,0.300280,NaN,2954
2,elasticnet,cv,0.071386,0.006796,0.104476,0.020864,0.479569,0.240773,0.053952,0.005248,1.441066,1.368479,12595
3,elasticnet,test,0.067844,NaN,0.091448,NaN,0.657164,NaN,0.050495,NaN,0.493434,NaN,2954
4,gradient_boosting,cv,0.062382,0.012029,0.084510,0.018029,0.657959,0.136321,0.047395,0.010006,0.381850,0.058402,12595
5,gradient_boosting,test,0.046555,NaN,0.063060,NaN,0.836977,NaN,0.035665,NaN,0.304866,NaN,2954
6,hist_gradient_boosting,cv,0.064248,0.012609,0.087939,0.019707,0.631675,0.147331,0.047154,0.008476,0.379468,0.058548,12595
7,hist_gradient_boosting,test,0.045079,NaN,0.063165,NaN,0.836432,NaN,0.032340,NaN,0.359945,NaN,2954
8,random_forest,cv,0.063569,0.014572,0.087544,0.022487,0.629167,0.172561,0.046364,0.009150,0.405437,0.068192,12595
9,random_forest,test,0.049782,NaN,0.068254,NaN,0.809018,NaN,0.037819,NaN,0.447436,NaN,2954


### 2. Metrics Comparison
Charts to compare MAE/RMSE/R² by model and split.

In [10]:
fig = px.bar(
    summary_df,
    x="model",
    y="mae_mean",
    color="split",
    error_y="mae_std",
    title="MAE medio por modelo y split",
)
fig.show()

fig = px.bar(
    summary_df,
    x="model",
    y="rmse_mean",
    color="split",
    error_y="rmse_std",
    title="RMSE medio por modelo y split",
)
fig.show()

fig = px.bar(
    summary_df,
    x="model",
    y="r2_mean",
    color="split",
    error_y="r2_std",
    title="R² medio por modelo y split",
)
fig.show()

### 3. Residuals and scatters (test)

We inspect how each model performs on the test set.

In [11]:
pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]

scatter_title = f"Dispersión {target_label} real vs predicho (test)"
fig = px.scatter(
    pred_df,
    x="y_true",
    y="y_pred",
    color="model",
    title=scatter_title,
    labels={"y_true": f"{target_label} real", "y_pred": f"{target_label} predicho"},
)
fig.add_trace(
    go.Scatter(
        x=[pred_df.y_true.min(), pred_df.y_true.max()],
        y=[pred_df.y_true.min(), pred_df.y_true.max()],
        mode="lines",
        name="Ideal",
    )
)
fig.show()

fig = px.box(
    pred_df,
    x="model",
    y="residual",
    title=f"Distribución de residuos por modelo (test) - {target_label}",
)
fig.show()

### 4. Feature Importance




In [12]:
import joblib
import numpy as np

available_models = sorted(summary_df["model"].unique())
for model_name in available_models:
    model_path = models_dir / f"{model_name}_best.joblib"
    if not model_path.exists():
        print(f"Model artifact not found for {model_name}.")
        continue

    pipeline = joblib.load(model_path)
    model = pipeline.named_steps["model"]

    importances = getattr(model, "feature_importances_", None)
    if importances is None:
        coef = getattr(model, "coef_", None)
        if coef is not None:
            importances = np.abs(np.ravel(coef))
        else:
            print(f"{model_name} does not expose feature importances or coefficients.")
            continue

    feature_names = (
        feature_columns
        if feature_columns is not None and len(feature_columns) == len(importances)
        else [f"f{i}" for i in range(len(importances))]
    )

    fi = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(20)
    )

    px.bar(fi, x="feature", y="importance", title=f"Top features ({model_name})").show()


hist_gradient_boosting does not expose feature importances or coefficients.
